In [ ]:
from argparse import ArgumentParser
import os
import sys
import time

import contextily as ctx
import dask.dataframe as dd
import dask_geopandas as dg
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
# import rasterio
# import sqlite3
import shapely
import xarray as xr

# from matplotlib import colors
from pandarallel import pandarallel
from scipy import stats

sys.path.insert(1, '../../narrow_rivers_PIXC/scripts/')
from reaches import *
from utils import *

In [ ]:
def calc_az(line):
    '''
    orthogonal, 09/11/2025 thanks to Ellie
    https://agupubs.onlinelibrary.wiley.com/doi/10.1002/2016GL072089
    '''
    ## Set-up
    # Extract coordinates
    x_coords = [coord[0] for coord in line.coords]
    y_coords = [coord[1] for coord in line.coords]
    coords = np.array([x_coords, y_coords]).T
    
    # Center the data
    x_mean = np.mean(x_coords)
    y_mean = np.mean(y_coords)
    centered = coords - [x_mean, y_mean]

    ## Perform regression
    # PCA for orthogonal regression
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    principal_direction = eigvecs[:, np.argmax(eigvals)]
    
    # Calculate azimuth from principal direction
    azimuth = math.degrees(math.atan2(principal_direction[1],
                                      principal_direction[0]))
    if azimuth < 0:
        azimuth += 360
    
    return azimuth

In [ ]:
def weightArea(df):
    '''Weight the land_near_water and water_near_land pixels
    by the water fraction, as per pg. 12 of the ATBD_L2_HR_RiverSP'''
    if df['klass'] == 2 or df['klass'] == 3:
        return df['water_frac'] * df['pixel_area']
    else:
        return df['pixel_area']

### Parse arguments

In [ ]:
# FOR NOW, SET
width_set = 'mean'

# Control flow
if width_set == 'mean':
    width = 'WidthM'
    binn = 'Bin'
elif width_set == 'min':
    width = 'WidthM_Min'
    binn = 'Bin_Min'
elif width_set == 'max':
    width = 'WidthM_Max'
    binn = 'Bin_Max'
else:
    print('Invalid width option specified, exiting.')
    # sys.exit()

In [ ]:
cpus = int(os.environ.get('SLURM_CPUS_PER_TASK'))
cpus_per_task = cpus if cpus < 65 else 1

huc2 = '01' ### SET THIS

mdata_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'
nhd_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/'
swot_dir = f'/nas/cee-ice/data/SWOT/PIXC_vD'

### Pixel Cloud

In [ ]:
file_name = 'SWOT_L2_HR_PIXC_041_298_080L_20251112T102954_20251112T103005_PID0_02.nc'

# Get data for this tile
granule_name = file_name[:-3]
tile_name = file_name[20:28]
pass_num = int(file_name[20:23])

print(granule_name)

#### Read in PIXC

In [ ]:
# Set PIXC filepath
pixc_path = os.path.join(swot_dir, file_name)

# Read in pixel group
ds_PIXC = xr.open_dataset(filename_or_obj=pixc_path,
                          group='pixel_cloud', engine='h5netcdf')

#### Now, true mask for analysis

In [ ]:
# Make mask
mask = bitwiseMask(ds_PIXC)

if mask.shape[0] == 0:
    print('This granule has no pixels after masking, exiting.')
    # sys.exit(1)

# Set desired data vars

variables = ['cross_track', 'water_frac',
             'pixel_area', 'height', 'geoid', 'solid_earth_tide',
             'load_tide_fes', 'pole_tide', 'prior_water_prob', 
             'phase_noise_std', 'dheight_dphase',
             'classification_qual', 'geolocation_qual', 'sig0_qual',
             'classification']

# Convert PIXC to GeoDataFrame
gdf_PIXC = makeGDF(ds=ds_PIXC, mask=mask, data_vars=variables)

# Get bounds of PIXC tile
pseudo_bounds = gdf_PIXC.total_bounds

gdf_PIXC = gdf_PIXC.drop(columns=['latitude', 'longitude'])

In [ ]:
## Flag as-in RiverSP
gdf_PIXC['geo_qual_suspect'] = 0
gdf_PIXC['geo_qual_suspect'] = \
np.where((gdf_PIXC['geolocation_qual'] >=2**7),
         1, gdf_PIXC['geo_qual_suspect'])

gdf_PIXC['class_qual_suspect'] = 0
gdf_PIXC['class_qual_suspect'] = \
np.where((gdf_PIXC['classification_qual'] >=2**7),
         1, gdf_PIXC['class_qual_suspect'])

gdf_PIXC['sig0_qual_suspect'] = 0
gdf_PIXC['sig0_qual_suspect'] = \
np.where((gdf_PIXC['sig0_qual'] >=2**7),
         1, gdf_PIXC['sig0_qual_suspect'])

gdf_PIXC = gdf_PIXC.drop(columns=['geolocation_qual',
                                  'classification_qual',
                                  'sig0_qual'])

In [ ]:
## Make indicator for wse aggregation
# (don't use lane_near_water or dark_water)
wse_klass = [3.0, 4.0, 6.0, 7.0]

gdf_PIXC['agg_wse'] = 0
gdf_PIXC.loc[gdf_PIXC['klass'].isin(wse_klass), 'agg_wse'] = 1

In [ ]:
# Calculate wse
gdf_PIXC['wse'] = (gdf_PIXC.height
                   - gdf_PIXC.geoid
                   - gdf_PIXC.solid_earth_tide
                   - gdf_PIXC.load_tide_fes
                   - gdf_PIXC.pole_tide)

gdf_PIXC = gdf_PIXC.drop(columns=['height', 'geoid', 'solid_earth_tide',
                                  'load_tide_fes', 'pole_tide'])

#### Find correct HUC4s

In [ ]:
### NHDPlus HR
## Find correct HUC4s
# Read in tile and HUC4 intersection data
dtype_dic= {'tile': str, 'huc4': str, 'coverage': float}
tile_huc4 = pd.read_csv(os.path.join(mdata_dir,
                                    'huc4_swot_science_tiles.csv'),
                        dtype=dtype_dic)

In [ ]:
# Make list of HUC4s that intersect the tile
huc4s = list(tile_huc4[(tile_huc4['tile'] == tile_name)]['huc4'])
# Limit to the current HUC2
huc4s = [x for x in huc4s if x.startswith(huc2)]

### Nadir track

In [ ]:
# Get single pixel for selecting correct nadir segment
pixel_pt = gdf_PIXC.iloc[0].geometry

In [ ]:
# Find correct nadir segment and return its geometry
nadir_segment_ln = findNadir(pass_num=pass_num, pixel_pt=pixel_pt)

In [ ]:
del pixel_pt

### Find azimuth of nadir

In [ ]:
az_nadir = calc_az(line=nadir_segment_ln)

### Find coverage

#### Calculate the weighted area

In [ ]:
gdf_PIXC_cov = gdf_PIXC[['points', 'klass', 'water_frac', 'pixel_area',
                         # 'NHDPlusID',
                         'geometry'
                        ]].copy()

#### Make pseudo pixels

In [ ]:
# Set along-track pixel resolution
azimuth_res = 22 # meters

In [ ]:
pandarallel.initialize(nb_workers=cpus_per_task)

In [ ]:
# Make pseudo pixels
start = time.time()
gdf_PIXC_cov['pseudo_pixel'] = gdf_PIXC_cov.parallel_apply(
    user_defined_function=makePseudoPixels,
    args=(nadir_segment_ln, azimuth_res), axis=1) # args are segment_ln, azimuth_res
end = time.time()
print(end - start)

In [ ]:
gdf_PIXC_cov = gdf_PIXC_cov.rename(
    columns={'geometry': 'pixel_centroid'}
).set_geometry('pseudo_pixel').set_crs(epsg=3857)

In [ ]:
gdf_PIXC_cov = gdf_PIXC_cov.drop(columns=[
    # 'NHDPlusID', 
    'pixel_centroid'])

In [ ]:
# # Clipped to corridor
# gdf_PIXC_cov.to_parquet(f'/nas/cee-water/cjgleason/fiona/for_presentations/2025_SWOT_ST/data/{granule_name}_pseudo.parquet')

# Clipped to corridor, OLD
gdf_PIXC_cov.to_parquet(f'/nas/cee-water/cjgleason/fiona/for_presentations/2025_AGU/data/{granule_name}_pseudo.parquet')

# # Unclipped 
# gdf_PIXC_cov.to_parquet(f'/nas/cee-water/cjgleason/fiona/for_presentations/2025_SWOT_ST/data/{granule_name}_pseudo_full.parquet')